In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

In [2]:
class DataLoader:
    
    @staticmethod
    def cargar_desde_csv(ruta_archivo):
        df = pd.read_csv(ruta_archivo)
        
        if 'Fecha' in df.columns and 'Hora' in df.columns:
            df['timestamp'] = pd.to_datetime(
                df['Fecha'] + ' ' + df['Hora'], 
                format='%d/%m/%y %H:%M'
            )
            df.set_index('timestamp', inplace=True)
            df.rename(columns={'Temp': 'temperatura', 'Hum': 'humedad'}, inplace=True)
            df.drop(['Fecha', 'Hora'], axis=1, inplace=True)
        
        return df

In [3]:
class StatisticalAnalyzer:
    
    @staticmethod
    def calcular_resumen(serie):
        stats = serie.describe()
        
        return {
            'Count': int(stats['count']),
            'Mean': round(stats['mean'], 2),
            'Std': round(stats['std'], 2),
            'Min': round(stats['min'], 2),
            '25%': round(stats['25%'], 2),
            '50%': round(stats['50%'], 2),
            '75%': round(stats['75%'], 2),
            'Max': round(stats['max'], 2)
        }
    
    @staticmethod
    def analizar_dataframe(df, columnas):
        resultados = {}
        for columna in columnas:
            if columna in df.columns:
                resultados[columna] = StatisticalAnalyzer.calcular_resumen(df[columna])
        return resultados

In [4]:
class ARIMAForecaster:
    
    def __init__(self, orden=(1, 1, 1)):
        self.orden = orden
        self.modelo_fit = None
        self.serie_original = None
    
    def entrenar(self, serie):
        self.serie_original = serie.values
        modelo = ARIMA(serie, order=self.orden)
        self.modelo_fit = modelo.fit()
        return self
    
    def pronosticar(self, pasos=6):
        if self.modelo_fit is None:
            raise ValueError("Modelo no entrenado. Ejecuta entrenar() primero")
        
        pronostico = self.modelo_fit.forecast(steps=pasos)
        return pronostico
    
    def obtener_metricas(self):
        if self.modelo_fit is None:
            return None
        
        return {
            'AIC': round(self.modelo_fit.aic, 2),
            'BIC': round(self.modelo_fit.bic, 2),
            'Log-Likelihood': round(self.modelo_fit.llf, 2)
        }
    
    def obtener_parametros(self):
        if self.modelo_fit is None:
            return None
        
        return {nombre: round(valor, 6) 
                for nombre, valor in self.modelo_fit.params.items()}
    
    def obtener_residuos(self):
        if self.modelo_fit is None:
            return None
        
        residuos = self.modelo_fit.resid
        return {
            'media': round(np.mean(residuos), 6),
            'desv_std': round(np.std(residuos), 6),
            'valores': residuos
        }

In [5]:
class ReportGenerator:
    
    @staticmethod
    def exportar_a_txt(ruta_archivo, df, estadisticas, pronosticos):
        with open(ruta_archivo, 'w', encoding='utf-8') as f:
            f.write("REPORTE DE ANALISIS - TERMOMETRO DE HUMEDAD Y TEMPERATURA\n\n")
            
            f.write(f"Fecha del reporte: {pd.Timestamp.now().strftime('%d/%m/%Y %H:%M:%S')}\n")
            f.write(f"Registros analizados: {len(df)}\n")
            f.write(f"Periodo de medicion: {df.index.min().strftime('%d/%m/%Y %H:%M')} a {df.index.max().strftime('%d/%m/%Y %H:%M')}\n\n")
            
            f.write("MEDICIONES\n")
            
            stats_temp = estadisticas['temperatura']
            stats_hum = estadisticas['humedad']
            
            for key in stats_temp.keys():
                f.write(f"{key:8s}: {stats_temp[key]} C")
                f.write(f"        {key:8s}: {stats_hum[key]} %\n")
            
            f.write("\nPRONOSTICOS\n")
            f.write(f"{'Paso':<8} {'Fecha y Hora':<25} {'Temperatura':<15} {'Humedad':<15}\n")
            
            ultimo_timestamp = df.index.max()
            
            for i in range(len(pronosticos['temperatura'])):
                tiempo_futuro = ultimo_timestamp + pd.Timedelta(minutes=(i+1)*5)
                fecha_hora = tiempo_futuro.strftime('%d/%m/%Y %H:%M')
                temp_val = pronosticos['temperatura'][i]
                hum_val = pronosticos['humedad'][i]
                f.write(f"{i+1:<8} {fecha_hora:<25} {temp_val:.2f} C{'':<9} {hum_val:.2f} %\n")

In [6]:
class AnalisisTemperaturaHumedad:
    
    def __init__(self, ruta_archivo):
        self.ruta_archivo = ruta_archivo
        self.df = None
        self.estadisticas = {}
        self.pronosticos = {}
        self.modelos = {}
    
    def cargar_datos(self):
        self.df = DataLoader.cargar_desde_csv(self.ruta_archivo)
        return self
    
    def calcular_estadisticas(self, columnas=['temperatura', 'humedad']):
        self.estadisticas = StatisticalAnalyzer.analizar_dataframe(self.df, columnas)
        return self
    
    def pronosticar(self, configuracion=None):
        if configuracion is None:
            configuracion = {
                'temperatura': (1, 1, 1),
                'humedad': (1, 1, 1)
            }
        
        for variable, orden in configuracion.items():
            if variable in self.df.columns:
                modelo = ARIMAForecaster(orden)
                modelo.entrenar(self.df[variable])
                self.pronosticos[variable] = modelo.pronosticar(pasos=6)
                self.modelos[variable] = modelo
        
        return self
    
    def exportar(self, ruta_salida='resultados_analisis.txt'):
        ReportGenerator.exportar_a_txt(
            ruta_salida, 
            self.df, 
            self.estadisticas, 
            self.pronosticos
        )
        print(f"Archivo generado: '{ruta_salida}'")
        return self
    
    def ejecutar_analisis_completo(self, exportar=True):
        self.cargar_datos()
        self.calcular_estadisticas()
        self.pronosticar()
        
        if exportar:
            self.exportar()
        
        return {
            'dataframe': self.df,
            'estadisticas': self.estadisticas,
            'pronosticos': self.pronosticos,
            'modelos': self.modelos
        }

In [7]:
analisis = AnalisisTemperaturaHumedad(r"C:\Users\Alberto Marquez\Desktop\ascii\datos_sd_D.txt")
resultados = analisis.ejecutar_analisis_completo()

Archivo generado: 'resultados_analisis.txt'
